# CARec with cached item embeddings

- Load one embedding condition and the saved interaction splits.
- Product text and sampling belong in [data_exploration.ipynb](data_exploration.ipynb). Encoder inference belongs in [carec_encoder_comparison.ipynb](carec_encoder_comparison.ipynb).


## What CARec already implements

- Build a user/product graph from training interactions and aggregate neighbours with LightGCN.
- Phase 1 learns the user embedding table while keeping cached product vectors fixed and bypassing the MLP.
- Phase 2 freezes the user table and learns an MLP that transforms the product vectors. Graph propagation still runs.
- Alignment brings interacting users and products closer; uniformity prevents collapse. The text encoder stays frozen.


In [11]:
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "requirements.txt").is_file()
)
DATASET_ID = "electronics_carec_035_seed2023"
PREPARED_DIR = PROJECT_ROOT / "data/processed" / DATASET_ID
EMBEDDING_NAME = "all-MiniLM-L6-v2.npz"
EMBEDDING_PATH = PROJECT_ROOT / "artifacts/embeddings" / DATASET_ID / EMBEDDING_NAME


## Load item embeddings

- Keep each vector paired with its original product ID.
- Change `EMBEDDING_NAME` to select another cached condition.


In [12]:
with np.load(EMBEDDING_PATH, allow_pickle=False) as saved:
    embedding_item_ids = saved["item_ids"]
    item_embeddings = saved["embeddings"]
    embedding_model_id = str(saved["model_id"])
    embedding_prompt = str(saved["prompt"])

print(embedding_model_id)
print(f"Embeddings: {item_embeddings.shape}")
print(f"Prompt: {embedding_prompt!r}")


sentence-transformers/all-MiniLM-L6-v2
Embeddings: (34220, 384)
Prompt: ''


## Load interaction splits

- Keep user and product IDs as strings.
- Only `train` interactions may enter the graph. Validation, test, and cold interactions stay held out.


In [13]:
partitions = {}
for name in ("train", "valid", "test", "cold"):
    partitions[name] = pd.read_csv(
        PREPARED_DIR / f"{name}.tsv.gz",
        sep="\t",
        dtype={"user_id:token": str, "item_id:token": str},
        float_precision="round_trip",
    )
    print(f"{name}: {len(partitions[name]):,} interactions")

train = partitions["train"]
valid = partitions["valid"]
test = partitions["test"]
cold = partitions["cold"]


train: 452,606 interactions
valid: 14,853 interactions
test: 133,274 interactions
cold: 23,163 interactions


## 1. Prepare files that RecBole understands

- RecBole is the external library that CARec uses to handle recommendation datasets.
- It reads interaction files named `<dataset>.train.inter`, `<dataset>.valid.inter`, and so on.
- Write copies of our existing splits under `artifacts/recbole/`. Keep the saved split boundaries and repeated interactions.
- The `.item` file lists every product, including cold products. Vectors stay in the loaded NumPy matrix for now.
- Export only the two ID fields and timestamp. `source_row` is an audit column without a RecBole type suffix, so it stays out of these files.


In [14]:
RECBOLE_ROOT = PROJECT_ROOT / "artifacts/recbole"
RECBOLE_DIR = RECBOLE_ROOT / DATASET_ID
RECBOLE_DIR.mkdir(parents=True, exist_ok=True)

interaction_columns = ["user_id:token", "item_id:token", "timestamp:float"]

for split_name, interactions in partitions.items():
    split_path = RECBOLE_DIR / f"{DATASET_ID}.{split_name}.inter"
    recbole_interactions = interactions[interaction_columns]
    recbole_interactions.to_csv(split_path, sep="\t", index=False)

product_catalogue = pd.DataFrame({"item_id:token": embedding_item_ids})
product_catalogue.to_csv(
    RECBOLE_DIR / f"{DATASET_ID}.item",
    sep="\t",
    index=False,
)
print(f"RecBole input files: {RECBOLE_DIR}")

RecBole input files: /Users/matouskovar/FIT/NI-PML/artifacts/recbole/electronics_carec_035_seed2023


## 2. Import the dataset tools

- `Config` holds RecBole settings, such as file names and column names.
- `Dataset` reads those files and assigns integer IDs to users and products. [Library implementation](https://github.com/RUCAIBox/RecBole/blob/v1.1.1/recbole/data/dataset/dataset.py).
- `load_carec()` imports our pinned CARec code. We use its model class in the configuration; we do not instantiate the model here.


In [15]:
import sys

from benchmarks.carec import load_carec
from recbole.config import Config
from recbole.data.dataset import Dataset

carec = load_carec()


## 3. Tell RecBole which files and fields to use

- `data_path` is the parent directory. RecBole appends `DATASET_ID` itself.
- `benchmark_filename` tells RecBole to keep our four saved splits, rather than create a new split.
- `load_col` lists field names without the `:token` or `:float` suffix.
- Disable numeric normalization. Keep the values we prepared. Dataset preparation uses CPU; training device selection comes later.


In [16]:
dataset_settings = {
    "data_path": str(RECBOLE_ROOT),
    "benchmark_filename": ["train", "valid", "test", "cold"],
    "field_separator": "\t",
    "USER_ID_FIELD": "user_id",
    "ITEM_ID_FIELD": "item_id",
    "TIME_FIELD": "timestamp",
    "load_col": {
        "inter": ["user_id", "item_id", "timestamp"],
        "item": ["item_id"],
    },
    "normalize_all": False,
    "use_gpu": False,
}

config = Config(
    model=carec.CollabContex,
    dataset=DATASET_ID,
    config_dict=dataset_settings,
)


## 4. Create the dataset and inspect its product IDs

- `Dataset(config)` loads the files and builds a shared ID vocabulary for all four splits.
- `field2id_token["item_id"]` translates a RecBole integer ID back to the original product ID.
- RecBole reserves ID `0` for `[PAD]`. This is a placeholder, not a real product.
- This dataset still contains all four splits. Later, only its training split may be passed to the graph model.


In [19]:
recbole_dataset = Dataset(config)
recbole_product_ids = recbole_dataset.field2id_token["item_id"]

print(f"Products including padding: {recbole_dataset.item_num:,}")
print(f"ID 0: {recbole_product_ids[0]}")
print(f"First real product: {recbole_product_ids[1]}")

for split_name, row_count in zip(config["benchmark_filename"], recbole_dataset.file_size_list):
    print(f"{split_name}: {row_count:,} interactions")


Products including padding: 34,221
ID 0: [PAD]
First real product: B00C7NSIO8
train: 452,606 interactions
valid: 14,853 interactions
test: 133,274 interactions
cold: 23,163 interactions


/Users/matouskovar/FIT/NI-PML/.venv/lib/python3.11/site-packages/recbole/data/dataset/dataset.py:638: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  feat[field].fillna(value=0, inplace=True)
/Users/matouskovar/FIT/NI-PML/.venv/lib/python3.11/site-packages/recbole/data/dataset/dataset.py:640: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on

## 5. Find each product's row in our saved embeddings

- Our cache has its own row order. Row `10` need not be RecBole product ID `10`.
- Build a dictionary from original product ID to its row in `item_embeddings`.
- `enumerate` gives us both the row number and the product ID.


In [18]:
embedding_row_by_product = {}
for embedding_row, product_id in enumerate(embedding_item_ids):
    embedding_row_by_product[product_id] = embedding_row

assert len(embedding_row_by_product) == len(embedding_item_ids), "Duplicate product IDs in cache"


## 6. Arrange vectors in RecBole order

- Allocate one row per RecBole product ID, including padding.
- Leave row `0` as zeros. For each real product, copy its cached vector into its RecBole row.
- The values stay unchanged. We only reorder rows and add the padding row.


In [21]:
embedding_dimension = item_embeddings.shape[1]
recbole_item_embeddings = np.zeros(
    (recbole_dataset.item_num, embedding_dimension),
    dtype=np.float32,
)

assert recbole_product_ids[0] == "[PAD]"
assert recbole_dataset.item_num == len(embedding_item_ids) + 1

for recbole_id in range(1, recbole_dataset.item_num):
    product_id = recbole_product_ids[recbole_id]
    embedding_row = embedding_row_by_product[product_id]
    recbole_item_embeddings[recbole_id] = item_embeddings[embedding_row]

print(f"Aligned embeddings: {recbole_item_embeddings.shape}")


Aligned embeddings: (34221, 384)


## 7. Inspect a few assignments

- Compare RecBole's integer ID, the original product ID, and the cache row.
- `same_vector` compares the copied vector with its source. Each displayed value should be `True`.
- For MiniLM, the aligned matrix should have 34,221 rows and 384 columns. The extra row is padding.


In [22]:
from IPython.display import display

preview_rows = []
for recbole_id in range(1, min(5, recbole_dataset.item_num)):
    product_id = recbole_product_ids[recbole_id]
    embedding_row = embedding_row_by_product[product_id]
    preview_rows.append({
        "recbole_id": recbole_id,
        "product_id": product_id,
        "cache_row": embedding_row,
        "same_vector": np.array_equal(
            recbole_item_embeddings[recbole_id], item_embeddings[embedding_row],
        ),
    })

display(pd.DataFrame(preview_rows))
print("Padding vector, first six values:", recbole_item_embeddings[0, :6])


,recbole_id,product_id,cache_row,same_vector
0,1,B00C7NSIO8,18133,True
1,2,B00AWRUIY4,17004,True
2,3,B005F778JO,12043,True
3,4,B009NB8WRU,16209,True


Padding vector, first six values: [0. 0. 0. 0. 0. 0.]


## 8. Attach the vectors to the dataset

- Add the aligned vectors as the product feature `item_emb`, which CARec expects.
- `FLOAT_SEQ` tells RecBole that each value is a vector of numbers. Register its source and dimension too.
- At this stage, `item_feat` is a pandas table. RecBole will convert the vectors to PyTorch tensors when we build the splits in the next step.


In [23]:
from recbole.utils.enum_type import FeatureSource, FeatureType

recbole_dataset.set_field_property(
    "item_emb",
    FeatureType.FLOAT_SEQ,
    FeatureSource.ITEM,
    embedding_dimension,
)
recbole_dataset.item_feat["item_emb"] = list(recbole_item_embeddings)

display(recbole_dataset.item_feat[["item_id", "item_emb"]].head(3))


,item_id,item_emb
0,0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1,1,"[-0.15457232, 0.04671826, -0.07804264, 0.05850..."
2,2,"[-0.063427605, 0.062324584, 0.04157299, -0.001..."


## 9. Separate the four saved splits

- `build()` returns datasets in the order from `benchmark_filename`: train, valid, test, cold. It keeps the saved split boundaries.
- It also converts pandas tables into RecBole `Interaction` objects, which store PyTorch tensors. The `item_emb` vectors become a float tensor.
- Every split shares the same ID mapping and product features. Only `train_dataset` will supply graph edges.
- To repeat this step, recreate `Dataset(config)` and rerun the alignment and attachment cells first; `build()` changes the tables in place.


In [24]:
train_dataset, valid_dataset, test_dataset, cold_dataset = recbole_dataset.build()

for split_name, split_dataset in [
    ("train", train_dataset),
    ("valid", valid_dataset),
    ("test", test_dataset),
    ("cold", cold_dataset),
]:
    print(f"{split_name}: {len(split_dataset.inter_feat):,} interactions")

print("Item embedding tensor:", train_dataset.item_feat["item_emb"].shape)


train: 452,606 interactions
valid: 14,853 interactions
test: 133,274 interactions
cold: 23,163 interactions
Item embedding tensor: torch.Size([34221, 384])


## 10. Configure the CARec model

- Match user vectors and MLP dimensions to the selected item embeddings.
- Keep the original runner's settings: two LightGCN propagation layers and an MLP with two configured hidden widths and dropout 0.2.
- Set the initialization seed so the same configuration can be repeated. The current device is CPU for this inspection; training device selection comes later.


In [25]:
config["seed"] = 2023
config["encoder"] = "LightGCN"
config["num_layers"] = 2
config["embedding_size"] = embedding_dimension
config["mlp"] = {
    "input_dim": embedding_dimension,
    "hidden_dims": [embedding_dimension, embedding_dimension],
    "output_dim": embedding_dimension,
    "num_layers": 2,
    "dropout": 0.2,
}


## 11. Create the original CARec model - IMPORTANT 

- Pass only `train_dataset`. Its interactions become the graph edges; held-out interactions do not enter the graph.
- The constructor reads `item_emb` and **initializes user vectors and MLP weights**. These new parameters have not learned anything yet.
- the user embeddings are random for now

In [27]:
from recbole.utils import init_seed

init_seed(config["seed"], config["reproducibility"])
carec_model = carec.CollabContex(config, train_dataset)
carec_model = carec_model.to(config["device"])


## 12. Inspect the model inputs and graph

- Each training interaction contributes two directed graph entries, user to product and product to user. Repeated interactions remain repeated entries.
- Embedding tables include the reserved padding row. The real user/product counts below exclude it.
- This step constructs the graph and model only. Graph propagation and training come next.


In [28]:
graph_encoder = carec_model.encoder

print(f"Users excluding padding: {graph_encoder.n_users - 1:,}")
print(f"Products excluding padding: {graph_encoder.n_items - 1:,}")
print(f"Training interactions: {len(train_dataset.inter_feat):,}")
print(f"Directed graph entries: {graph_encoder.edge_index.shape[1]:,}")
print("User embedding table:", graph_encoder.user_embedding.weight.shape)
print("Cached item vectors:", graph_encoder.item_embedding.shape)
print("Device:", config["device"])


Users excluding padding: 81,520
Products excluding padding: 34,220
Training interactions: 452,606
Directed graph entries: 905,212
User embedding table: torch.Size([81521, 384])
Cached item vectors: torch.Size([34221, 384])
Device: cpu


## 13. One forward pass without training

- Select the first four training interactions and request their user/product vectors.
- Use phase 1: the MLP is bypassed and cached product vectors enter LightGCN directly.
- The original model propagates over the entire training graph, then selects the four requested pairs. This is not a four-edge subgraph.
- `eval()` selects evaluation mode; `no_grad()` avoids storing gradients. No optimizer is used, so model weights stay unchanged.


In [29]:
import torch

carec_model.set_item_tutoring_phase()
carec_model.eval()

example_interactions = train_dataset.inter_feat[:4]
example_user_ids = example_interactions[config["USER_ID_FIELD"]].to(config["device"])
example_item_ids = example_interactions[config["ITEM_ID_FIELD"]].to(config["device"])


- CARec's `forward` returns normalized user and product vectors after graph propagation. Their norms should be close to 1.
- MiniLM should give two matrices of shape `[4, 384]`. User parameters are still at their initial values; this does not measure recommendation quality.


In [ ]:
with torch.no_grad():
    user_vectors, product_vectors = carec_model(example_user_ids, example_item_ids)


print("User vectors:", user_vectors.shape)
print("Product vectors:", product_vectors.shape)
display(pd.DataFrame({
    "user_id": example_user_ids.cpu().numpy(),
    "item_id": example_item_ids.cpu().numpy(),
    "user_vector_norm": user_vectors.norm(dim=1).cpu().numpy(),
    "product_vector_norm": product_vectors.norm(dim=1).cpu().numpy(),
}))


User vectors: torch.Size([4, 384])
Product vectors: torch.Size([4, 384])


,user_id,item_id,user_vector_norm,product_vector_norm
0,1,1,1.0,1.0
1,1,2,1.0,1.0
2,1,3,1.0,1.0
3,1,4,1.0,1.0


## 14. Calculate the loss without training

- Sample 64 training interactions with a fixed seed. The previous four rows all belonged to one user; this batch should contain several users.
- Call the original CARec loss in phase 1. Alignment brings interacting user/product vectors closer. User uniformity discourages all user vectors from becoming the same.
- Graph propagation still uses the whole training graph. Only these 64 pairs enter the loss.
- A negative total loss is possible because the uniformity term can be negative. This initial value is not a recommendation metric.
- Keep evaluation mode and disable gradients. No weights are updated.


In [31]:
batch_size = 64
batch_generator = torch.Generator().manual_seed(config["seed"])
batch_indices = torch.randperm(len(train_dataset.inter_feat), generator=batch_generator)[:batch_size]
loss_batch = train_dataset.inter_feat[batch_indices].to(config["device"])

batch_user_count = loss_batch[config["USER_ID_FIELD"]].unique().numel()
if batch_user_count < 2:
    raise ValueError("User uniformity needs at least two distinct users in the batch.")

carec_model.set_item_tutoring_phase()
carec_model.eval()
with torch.no_grad():
    loss = carec_model.calculate_loss(loss_batch)

print(f"Interactions: {len(loss_batch)}")
print(f"Distinct users: {batch_user_count}")
print(f"Phase 1 loss: {loss.item():.6f}")


Interactions: 64
Distinct users: 64
Phase 1 loss: -1.293274


## 15. Train phase 1

- Start with one epoch over every training interaction, shuffled with seed 2023.
- Use Adam, learning rate 0.001, and batch size 1024, matching the upstream learning rate and batch size.
- Train the user embedding table. Cached item vectors stay fixed and the MLP stays bypassed.
- Each batch runs propagation over the entire training graph before calculating its loss. The current model is on CPU; the progress bar shows the actual speed.
- Print the average batch loss after each epoch. Recommendation quality needs separate validation.
- Rerunning this section continues from the current weights with a new optimizer. Rerun model construction first to start over.
- This is a first training run. Phase 2, validation, early stopping, and checkpoint saving come later.


In [32]:
from tqdm.auto import tqdm

phase1_epochs = 1
training_batch_size = 1024
learning_rate = 0.001

carec_model.set_item_tutoring_phase()
carec_model.train()
optimizer = torch.optim.Adam(
    carec_model.encoder.user_embedding.parameters(),
    lr=learning_rate,
)
training_generator = torch.Generator().manual_seed(config["seed"])
phase1_losses = []


In [33]:
for epoch in range(phase1_epochs):
    shuffled_indices = torch.randperm(
        len(train_dataset.inter_feat), generator=training_generator
    )
    total_loss = 0.0
    interaction_count = 0
    batches = tqdm(
        shuffled_indices.split(training_batch_size),
        desc=f"Phase 1, epoch {epoch + 1}/{phase1_epochs}",
    )

    for indices in batches:
        interaction_batch = train_dataset.inter_feat[indices].to(config["device"])
        if interaction_batch[config["USER_ID_FIELD"]].unique().numel() < 2:
            raise ValueError("User uniformity needs at least two distinct users.")

        optimizer.zero_grad(set_to_none=True)
        loss = carec_model.calculate_loss(interaction_batch)
        if not torch.isfinite(loss):
            raise ValueError("Training loss is not finite.")
        loss.backward()
        optimizer.step()

        batch_loss = loss.item()
        total_loss += batch_loss * len(interaction_batch)
        interaction_count += len(interaction_batch)
        batches.set_postfix(loss=f"{batch_loss:.4f}")

    average_loss = total_loss / interaction_count
    phase1_losses.append(average_loss)
    print(f"Epoch {epoch + 1}: average batch loss = {average_loss:.6f}")

carec_model.eval()


Phase 1, epoch 1/1: 100%|██████████| 442/442 [06:34<00:00,  1.12it/s, loss=-2.5687]

Epoch 1: average batch loss = -2.568197


CollabContex(
  (encoder): LGCNEncoder(
    (gcn_conv): LightGCNConv(384)
    (mlp): MLP(
      (mlp): Sequential(
        (0): Linear(in_features=384, out_features=384, bias=True)
        (1): ReLU()
        (2): Dropout(p=0.2, inplace=False)
        (3): Linear(in_features=384, out_features=384, bias=True)
        (4): ReLU()
        (5): Dropout(p=0.2, inplace=False)
        (6): Linear(in_features=384, out_features=384, bias=True)
      )
    )
    (user_embedding): Embedding(81521, 384)
  )
)